# 02 Modeling

In [ ]:
import joblib
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    fbeta_score,
    make_scorer,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    cross_validate,
)
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

from src.data_processor import (
    build_preprocessor,
    load_and_clean_data,
    load_and_split_data,
)


In [ ]:
RANDOM_STATE=42

## Loading data and preprocessor

Functions from `data_processor.py` are used for retrieving splitted data for modeling and preprocessor for pipelining.  

This is a classification problem, trying to predict **Attrition** from employees.  

`stratify` is set to `True` since there's a imbalanced proportion (~84/16) for employees that have attrited. 

In [ ]:
X_train, X_test, y_train, y_test = load_and_split_data(
    "../data/raw/HR-Employee-Attrition.csv",
    target_col="Attrition",
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=True,
)

In [ ]:
y_train.value_counts()

In [ ]:
y_test.value_counts()

In [ ]:
preprocessor = build_preprocessor()

## Baselining

For baselining, three classifiers are used:
1. `DummyClassifier`
2. `LogisticRegression`
3. `XGBClassifier`

For the three, class imbalance parameters are used, respectively:
1. `strategy="stratified"`
2. `class_weight="balanced"`
3. `scale_pos_weight`: This one is set to the ratio between negative examples and positive examples.

In [ ]:
dummy_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("clf", DummyClassifier(strategy="stratified", random_state=RANDOM_STATE))
])

lr_baseline = Pipeline([
    ("preprocessor", preprocessor),
    ("clf", LogisticRegression(class_weight="balanced", random_state=RANDOM_STATE))
])

scale_pos_weight = y_train.value_counts(normalize=True).iloc[0] / y_train.value_counts(normalize=True).iloc[1] 
xgb_baseline = Pipeline([
    ("preprocessor", preprocessor),
    ("clf", XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=RANDOM_STATE))
])

In [ ]:
%%time

f2_scorer = make_scorer(fbeta_score, beta=2)
scoring = {
    "balanced_accuracy": "balanced_accuracy",
    "average_precision": "average_precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "f2": f2_scorer,
}
skf = StratifiedKFold(n_splits=5)

cv_results = []

for model in (dummy_pipeline, lr_baseline, xgb_baseline):
    scores = cross_validate(
        model, X_train, y_train, scoring=scoring, cv=skf, return_train_score=True
    )
    scores["estimator"] = model.named_steps["clf"].__class__.__name__
    cv_results.append(scores)

In [ ]:
train_metrics = [
    "train_balanced_accuracy",
    "train_average_precision",
    "train_recall",
    "train_f1",
    "train_roc_auc",
    "train_f2",
]

test_metrics = [
    "test_balanced_accuracy",
    "test_average_precision",
    "test_recall",
    "test_f1",
    "test_roc_auc",
    "test_f2",
]

for res in cv_results:
    print(f"=== Model: {res['estimator']} ===\n")

    for metric in train_metrics:
        scores = res[metric]
        print(f"{metric}:")
        print(f"  Mean: {scores.mean():.4f}")
        print(f"  Std:  {scores.std():.4f}")
        print(f"  Folds: {[round(s, 4) for s in scores]}")
        print()

    for metric in test_metrics:
            scores = res[metric]
            print(f"{metric}:")
            print(f"  Mean: {scores.mean():.4f}")
            print(f"  Std:  {scores.std():.4f}")
            print(f"  Folds: {[round(s, 4) for s in scores]}")
            print()

These are the results for the three classifiers with cross-validation:

**<u>Test Metrics</u>**

| Model | Balanced Acc | Avg Precision | Recall | F1 | ROC AUC | F2 |
|---|---|---|---|---|---|---|
| DummyClassifier | 0.5054 ± 0.0170 | 0.1640 ± 0.0047 | 0.1579 ± 0.0288 | 0.1644 ± 0.0300 | 0.5054 ± 0.0170 | 0.1604 ± 0.0293 |
| LogisticRegression | 0.7612 ± 0.0473 | 0.6148 ± 0.0823 | 0.7526 ± 0.0906 | 0.5129 ± 0.0604 | 0.8277 ± 0.0533 | 0.6326 ± 0.0677 |
| XGBClassifier | 0.6625 ± 0.0399 | 0.5511 ± 0.0773 | 0.3737 ± 0.0770 | 0.4574 ± 0.0806 | 0.7893 ± 0.0487 | 0.4028 ± 0.0780 |

**<u>Train Metrics</u>**

| Model | Balanced Acc | Avg Precision | Recall | F1 | ROC AUC | F2 |
|---|---|---|---|---|---|---|
| DummyClassifier | 0.4776 ± 0.0120 | 0.1573 ± 0.0023 | 0.1263 ± 0.0206 | 0.1254 ± 0.0202 | 0.4776 ± 0.0120 | 0.1259 ± 0.0204 |
| LogisticRegression | 0.7972 ± 0.0165 | 0.6853 ± 0.0368 | 0.8066 ± 0.0255 | 0.5549 ± 0.0204 | 0.8813 ± 0.0117 | 0.6827 ± 0.0232 |
| XGBClassifier | 1.0000 ± 0.0 | 1.0000 ± 0.0 | 1.0000 ± 0.0 | 1.0000 ± 0.0 | 1.0000 ± 0.0 | 1.0000 ± 0.0 |

Clearly, `LogisticRegression` shows better results for all the metrics. `XGBClassifier` is overfitting, obtaining 1.0 in all the train metrics but worse results than `LogisticRegression` in test metrics.

---

## Hyperparameter tuning
### `LogisticRegression`
Firstly, `LogisticRegression` will be optimized using `f2` scoring. 

In [ ]:
%%time
lr = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("clf", LogisticRegression(solver="lbfgs", max_iter=1000, random_state=RANDOM_STATE)),
    ]
)

param_grid = {
    "clf__C": np.logspace(-3, 3, 7).tolist(),
    "clf__class_weight": [None, "balanced"],
}

lr_gs = GridSearchCV(lr, param_grid=param_grid, scoring=f2_scorer, cv=skf, n_jobs=4, verbose=1)
lr_gs.fit(X_train, y_train)

In [ ]:
lr_gs.best_params_, lr_gs.best_score_

In [ ]:
scores_lr = cross_validate(
        lr_gs.best_estimator_, X_train, y_train, scoring=scoring, cv=skf, return_train_score=True
)
print("TRAIN: ", scores_lr["train_f2"].mean(), scores_lr["train_f2"].std())
print("TEST: ", scores_lr["test_f2"].mean(), scores_lr["test_f2"].std())

Tuned model still preferred `class_weight="balanced"`, with a strong regularization `C=0.1`.

---

### `XGBClassifier`

In [ ]:
%%time
xgb = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("clf", XGBClassifier(random_state=RANDOM_STATE)),
    ]
)

param_distribs = {
    "clf__max_depth": np.arange(2, 11, 1).tolist(),
    "clf__min_child_weight": np.arange(0, 8, 1).tolist(),
    "clf__subsample": np.linspace(0.5, 1.0, 6).tolist(),
    "clf__colsample_bytree": np.linspace(0.5, 1.0, 6).tolist(),
    "clf__reg_alpha": [0, 0.01, 0.1, 1, 5, 10, 50],
    "clf__reg_lambda": [0, 0.01, 0.1, 1, 5, 10, 50],
    "clf__learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
    "clf__n_estimators": [200, 400, 600, 800],
}

xgb_rs = RandomizedSearchCV(
    xgb,
    param_distributions=param_distribs,
    n_iter=80,
    scoring=f2_scorer,
    cv=skf,
    n_jobs=4,
    verbose=1,
    random_state=RANDOM_STATE,
)
xgb_rs.fit(X_train, y_train)

In [ ]:
xgb_rs.best_params_, xgb_rs.best_score_

In [ ]:
scores_xgb = cross_validate(
        xgb_rs.best_estimator_, X_train, y_train, scoring=scoring, cv=skf, return_train_score=True
)
print("TRAIN: ", scores_xgb["train_f2"].mean(), scores_xgb["train_f2"].std())
print("TEST: ", scores_xgb["test_f2"].mean(), scores_xgb["test_f2"].std())

Using `RandomizedSearchCV`, two hyperparameters are at the boundaries of the search space: `max_depth` and `n_estimators`. These will be used in a `GridSearchCV` with other hyperparameters set to the best results found in `RandomizedSearchCV`.

In [ ]:
best_subsample = xgb_rs.best_params_["clf__subsample"]
best_reg_lambda = xgb_rs.best_params_["clf__reg_lambda"]
best_reg_alpha = xgb_rs.best_params_["clf__reg_alpha"]
best_min_child_weight = xgb_rs.best_params_["clf__min_child_weight"]
best_learning_rate = xgb_rs.best_params_["clf__learning_rate"]
best_colsample_bytree = xgb_rs.best_params_["clf__colsample_bytree"]

In [ ]:
%%time
xgb_v2 = Pipeline(
    [
        ("preprocessor", preprocessor),
        (
            "clf",
            XGBClassifier(
                random_state=RANDOM_STATE,
                subsample=best_subsample,
                reg_lambda=best_reg_lambda,
                reg_alpha=best_reg_alpha,
                min_child_weight=best_min_child_weight,
                learning_rate=best_learning_rate,
                colsample_bytree=best_colsample_bytree,
            ),
        ),
    ]
)

param_grid = {
    "clf__max_depth": np.arange(1, 6, 1).tolist(),
    "clf__n_estimators": [800, 1000, 1200, 1400],
}

xgb_gs = GridSearchCV(
    xgb_v2, param_grid=param_grid, scoring=f2_scorer, cv=skf, n_jobs=4, verbose=1
)
xgb_gs.fit(X_train, y_train)

In [ ]:
xgb_gs.best_params_, xgb_gs.best_score_

In [ ]:
scores_xgb_v2 = cross_validate(
        xgb_gs.best_estimator_, X_train, y_train, scoring=scoring, cv=skf, return_train_score=True
)
print("TRAIN: ", scores_xgb_v2["train_f2"].mean(), scores_xgb_v2["train_f2"].std())
print("TEST: ", scores_xgb_v2["test_f2"].mean(), scores_xgb_v2["test_f2"].std())

F2 score only improved ~0.0082 after `GridSearchCV` and still showing severe overfitting. 

---

### `RandomForest`

An additional classifier is going to be used for analysis; only `class_weight="balanced"` hyperparameter is used for fair comparison. 

In [ ]:
rf = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("clf", RandomForestClassifier(random_state=RANDOM_STATE, class_weight="balanced")),
    ]
)

scores_rf = cross_validate(
        rf, X_train, y_train, scoring=scoring, cv=skf, return_train_score=True
)
print("TRAIN: ", scores_rf["train_f2"].mean(), scores_rf["train_f2"].std())
print("TEST: ", scores_rf["test_f2"].mean(), scores_rf["test_f2"].std())

In [ ]:
%%time
rf_v2 = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("clf", RandomForestClassifier(criterion="gini", random_state=RANDOM_STATE)),
    ]
)

param_distribs = {
    "clf__n_estimators": [100, 300, 500, 700, 900, 1000],
    "clf__max_depth": np.arange(1, 10, 1).tolist(),
    "clf__max_features": ["sqrt", "log2", None],
    "clf__min_samples_split": np.arange(2, 10, 1).tolist(),
    "clf__class_weight": [None, "balanced", "balanced_subsample"],
}

rf_rs = RandomizedSearchCV(
    rf_v2,
    param_distributions=param_distribs,
    n_iter=80,
    scoring=f2_scorer,
    cv=skf,
    n_jobs=4,
    verbose=1,
    random_state=RANDOM_STATE,
)
rf_rs.fit(X_train, y_train)

In [ ]:
rf_rs.best_params_, rf_rs.best_score_

In [ ]:
scores_rf_v2 = cross_validate(
        rf_rs.best_estimator_, X_train, y_train, scoring=scoring, cv=skf, return_train_score=True
)
print("TRAIN: ", scores_rf_v2["train_f2"].mean(), scores_rf_v2["train_f2"].std())
print("TEST: ", scores_rf_v2["test_f2"].mean(), scores_rf_v2["test_f2"].std())

`RandomForest` has a higher score than `XGBClassifier` (0.534 vs 0.485) and less overfitting:
- `RandomForest` train/test gap: ~0.20
- `XGBClassifier` train/test gap: ~0.47

---

## Model comparison

With all models' results from cross-validation, same metrics from baselining will be used for analysis.

In [ ]:
final_cv_results = {
    "LogisticRegression": scores_lr, 
    "XGBClassifier": scores_xgb, 
    "XGBClassifier v2": scores_xgb_v2, 
    "RandomForest": scores_rf,
    "RandomForest v2": scores_rf_v2,
}

for name, res in final_cv_results.items():

    print(f"=== Model: {name} ===\n")

    for metric in train_metrics:
        results = res[metric]
        print(f"{metric}:")
        print(f"  Mean: {results.mean():.4f}")
        print(f"  Std:  {results.std():.4f}")
        print(f"  Folds: {[round(s, 4) for s in results]}")
        print()
    
    for metric in test_metrics:
        results = res[metric]
        print(f"{metric}:")
        print(f"  Mean: {results.mean():.4f}")
        print(f"  Std:  {results.std():.4f}")
        print(f"  Folds: {[round(s, 4) for s in results]}")
        print()

These are the results for the five tuned models with cross-validation:

**<u>Test Metrics</u>**

| Model | Balanced Acc | Avg Precision | Recall | F1 | ROC AUC | F2 |
|---|---|---|---|---|---|---|
| LogisticRegression | 0.7650 ± 0.0492 | 0.6215 ± 0.0808 | 0.7632 ± 0.0896 | 0.5157 ± 0.0620 | 0.8276 ± 0.0585 | 0.6389 ± 0.0691 |
| XGBClassifier | 0.7014 ± 0.0390 | 0.5837 ± 0.0982 | 0.4474 ± 0.0781 | 0.5306 ± 0.0709 | 0.8086 ± 0.0482 | 0.4768 ± 0.0751 |
| XGBClassifier v2 | 0.7051 ± 0.0407 | 0.5783 ± 0.0954 | 0.4579 ± 0.0842 | 0.5336 ± 0.0707 | 0.8037 ± 0.0492 | 0.4849 ± 0.0794 |
| RandomForest | 0.6439 ± 0.0234 | 0.5120 ± 0.0612 | 0.3263 ± 0.0488 | 0.4261 ± 0.0453 | 0.7878 ± 0.0564 | 0.3598 ± 0.0481 |
| RandomForest v2 | 0.7089 ± 0.0387 | 0.5128 ± 0.0673 | 0.5789 ± 0.0725 | 0.4802 ± 0.0528 | 0.7823 ± 0.0577 | 0.5341 ± 0.0598 |

**<u>Train Metrics</u>**

| Model | Balanced Acc | Avg Precision | Recall | F1 | ROC AUC | F2 |
|---|---|---|---|---|---|---|
| LogisticRegression | 0.7915 ± 0.0178 | 0.6948 ± 0.0288 | 0.7987 ± 0.0234 | 0.5477 ± 0.0240 | 0.8762 ± 0.0116 | 0.6749 ± 0.0245 |
| XGBClassifier | 0.9557 ± 0.0088 | 0.9924 ± 0.0028 | 0.9118 ± 0.0175 | 0.9525 ± 0.0099 | 0.9979 ± 0.0011 | 0.9277 ± 0.0146 |
| XGBClassifier v2 | 0.9711 ± 0.0048 | 0.9973 ± 0.0021 | 0.9421 ± 0.0097 | 0.9702 ± 0.0051 | 0.9993 ± 0.0006 | 0.9531 ± 0.0079 |
| RandomForest | 1.0000 ± 0.0 | 1.0000 ± 0.0 | 1.0000 ± 0.0 | 1.0000 ± 0.0 | 1.0000 ± 0.0 | 1.0000 ± 0.0 |
| RandomForest v2 | 0.8372 ± 0.0142 | 0.7367 ± 0.0144 | 0.7921 ± 0.0234 | 0.6597 ± 0.0232 | 0.9125 ± 0.0067 | 0.7331 ± 0.0218 |

F2 and recall metrics are very important for this case. False negatives are expensive, since not detecting potential attritions will be a waste in training and resources.  

`LogisticRegression` shows better results for these metrics (Recall=0.7632, F2=0.6389), with a low gap between train and test metrics (0.03, 0.03), indicating better generalization.  

`XGBClassifier` shows marginal improvements after second tuning (v2):
- Recall: +0.0105 (0.4474 → 0.4579)
- F2: +0.0081 (0.4768 → 0.4849)

However, this comes with a wider train/test gap:
- Recall gap: 0.4644 → 0.4842
- F2 gap: 0.4509 → 0.4682

`RandomForest` shows that tuning substantially reduced overfitting and improved performance:
- Recall gap: 0.6737 → 0.2132
- F2 gap: 0.6402 → 0.1990
- Test Recall: 0.3263 → 0.5789
- Test F2: 0.3598 → 0.5341

Despite this improvement, `LogisticRegression` remains the best-performing model for this use case, with the highest test Recall (0.7632) and F2 (0.6389) with the smallest train/test gap among the models, indicating the most reliable generalization to unseen data.

---

## Feature importance

Features from data will be sorted by its coefficient, according to the fitted model.

In [ ]:
final_model = lr_gs.best_estimator_

In [ ]:
pd.DataFrame(
    {
        "feat": final_model.named_steps["preprocessor"].get_feature_names_out(),
        "coef": final_model.named_steps["clf"].coef_.flatten(),
    }
).sort_values(by="coef", ascending=False)

**Top factors increasing attrition risk:**
- `OverTime` (+1.24) - strongest predictor; working overtime increases risk.
- `BusinessTravel` (+0.68) - more frequent travel is associated with higher attrition.
- `JobRole_Laboratory Technician` (+0.68) - this role shows higher attrition propensity than others.
- `YearsSinceLastPromotion` (+0.42) - longer time without promotion increases risk.
- `NumCompaniesWorked` (+0.38) - a history of more employers correlates with attrition risk.
- `MaritalStatus_Single` (+0.37) - single employees show higher risk.

**Top factors decreasing attrition risk:**
- `TotalWorkingYears` (-0.38) - greater seniority reduces attrition risk.
- `YearsWithCurrManager` (-0.37) - employees with more time with same manager are less likely to attrition.
- `EnvironmentSatisfaction` (-0.36) & `JobSatisfaction` (-0.34) - employees with higher satisfaction reduce risk.

**No contribution (coefficients near 0, negligible effect on predictions):**
- `Education` (-0.006)
- `Department_Human Resources` (0.002)
- `MonthlyRate` (0.024)
- `JobRole_Manager` (0.009)

---

## Final test 

In [ ]:
y_pred = final_model.predict(X_test)
y_proba = final_model.predict_proba(X_test)[:, 1]

test_metrics = {
    "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
    "average_precision": average_precision_score(y_test, y_proba),
    "recall": recall_score(y_test, y_pred),
    "f1": f1_score(y_test, y_pred),
    "f2": fbeta_score(y_test, y_pred, beta=2),
    "roc_auc": roc_auc_score(y_test, y_proba),
}

In [ ]:
for name, value in test_metrics.items():
    print(f"{name}: {value:.4f}")

**<u>LogisticRegression: CV vs Final Test</u>**

| Metric | CV Test (mean ± std) | Final Test | Difference |
|---|---|---|---|
| Balanced Acc | 0.7650 ± 0.0492 | 0.7205 | -0.0445 |
| Avg Precision | 0.6215 ± 0.0808 | 0.5730 | -0.0485 |
| Recall | 0.7632 ± 0.0896 | 0.6596 | -0.1036 |
| F1 | 0.5157 ± 0.0620 | 0.4697 | -0.0460 |
| ROC AUC | 0.8276 ± 0.0585 | 0.8082 | -0.0194 |
| F2 | 0.6389 ± 0.0691 | 0.5678 | -0.0711 |

Final test confirms the CV estimates, with all metrics falling within ~1 standard deviation of their cross-validated means. Variability is expected due to the small number of positive cases in the test set (n=47).

---

## Model export

In [ ]:
X, y = load_and_clean_data(
    "../data/raw/HR-Employee-Attrition.csv", target_col="Attrition"
)

In [ ]:
export_model = Pipeline(
    [
        ("preprocessor", preprocessor),
        (
            "clf",
            LogisticRegression(
                C=lr_gs.best_params_["clf__C"],
                class_weight=lr_gs.best_params_["clf__class_weight"],
                solver="lbfgs",
                max_iter=1000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)
export_model.fit(X, y)

In [ ]:
joblib.dump(export_model, "../models/attrition_pipeline.joblib")